# Data Engineer (итерация 1)

# Data Engineer Report

**Проект:** Бинарная классификация мошеннических вакансий (fake_job_postings).
**Бизнес-цель:** снизить ручную модерацию, защитить пользователей от скам-вакансий.
**Метрика-приоритет:** F1 / recall класса 1 при контроле precision.

**Вход:** `data/raw/fake_job_postings.csv`
**Выход:** `/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv`
**Target:** `fraudulent` — НЕ трогаем.

## План
1. Загрузка и базовый shape/NaN-чек.
2. Профиль: dtypes, доля NaN, распределение target, разделение на num / cat_low / cat_high / text.
3. Стратегия очистки:
   - парсинг `salary_range` → `salary_min` / `salary_max` (чтобы не терять сигнал до drop по >70% NaN);
   - drop колонок с >70% NaN;
   - числовые: clip по [1, 99] перцентилям, impute median (распределения скошены, есть выбросы);
   - категориальные < 20 уникальных → one-hot;
   - категориальные > 50 уникальных → frequency encoding;
   - текстовые (description/requirements/benefits/company_profile/title) → as_is (DS решит как фичеризовать);
   - target не трогаем.

## Фикс после QC-фидбека
- Детект категориальных через `pd.api.types.is_object_dtype` / `is_string_dtype` (раньше сравнение `== 'str'` давало пустой CAT_COLS).
- `salary_range` парсится в `salary_min`/`salary_max` ДО drop по NaN, чтобы сохранить сигнал.

In [ ]:
import pandas as pd
import numpy as np

DF = pd.read_csv("data/raw/fake_job_postings.csv")
print("shape:", DF.shape)
print("NaN per column:")
print(DF.isna().sum())

shape: (17880, 18)
NaN per column:
job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64


## Профиль данных

Смотрим dtypes, доли NaN, распределение target и разделяем колонки на группы:
- `NUM_COLS` — числовые;
- `TEXT_COLS` — длинные текстовые поля (фичеризует DS);
- `CAT_LOW` — категориальные с <20 уникальных → one-hot;
- `CAT_HIGH` — категориальные с >50 уникальных → frequency encoding;
- `CAT_MID` (20..50 уникальных) → frequency encoding как безопасный дефолт.

Важный фикс: детект object-колонок через `pd.api.types.is_object_dtype` / `is_string_dtype`, а не по строковому имени dtype.

In [ ]:
print("dtypes:")
print(DF.dtypes)
print("\nNaN ratio:")
print((DF.isna().mean().sort_values(ascending=False)).round(3))
print("\ntarget distribution (fraudulent):")
print(DF['fraudulent'].value_counts(dropna=False))
print("ratio:", DF['fraudulent'].mean())

TARGET = 'fraudulent'

# Текстовые колонки — длинные свободные тексты, отдаём DS как есть
TEXT_COLS = [c for c in ['title', 'company_profile', 'description', 'requirements', 'benefits']
             if c in DF.columns]

# Числовые
NUM_COLS = [c for c in DF.columns
            if c != TARGET
            and c not in TEXT_COLS
            and pd.api.types.is_numeric_dtype(DF[c])]

# Категориальные — всё, что object/string и не в тексте
CAT_COLS = [c for c in DF.columns
            if c != TARGET
            and c not in TEXT_COLS
            and c not in NUM_COLS
            and (pd.api.types.is_object_dtype(DF[c]) or pd.api.types.is_string_dtype(DF[c]))]

print("\nTEXT_COLS:", TEXT_COLS)
print("NUM_COLS:", NUM_COLS)
print("CAT_COLS:", CAT_COLS)

print("\nУникальных значений в категориальных:")
for c in CAT_COLS:
    print(f"  {c}: nunique={DF[c].nunique(dropna=True)}, NaN={DF[c].isna().sum()}")

CAT_LOW  = [c for c in CAT_COLS if DF[c].nunique(dropna=True) < 20]
CAT_HIGH = [c for c in CAT_COLS if DF[c].nunique(dropna=True) > 50]
CAT_MID  = [c for c in CAT_COLS if c not in CAT_LOW and c not in CAT_HIGH]

print("\nCAT_LOW  (<20 → one-hot):", CAT_LOW)
print("CAT_MID  (20..50 → frequency):", CAT_MID)
print("CAT_HIGH (>50 → frequency):", CAT_HIGH)

dtypes:
job_id                 int64
title                    str
location                 str
department               str
salary_range             str
company_profile          str
description              str
requirements             str
benefits                 str
telecommuting          int64
has_company_logo       int64
has_questions          int64
employment_type          str
required_experience      str
required_education       str
industry                 str
function                 str
fraudulent             int64
dtype: object

NaN ratio:
salary_range           0.840
department             0.646
required_education     0.453
benefits               0.403
required_experience    0.394
function               0.361
industry               0.274
employment_type        0.194
company_profile        0.185
requirements           0.151
location               0.019
description            0.000
job_id                 0.000
telecommuting          0.000
has_questions          0.000
has_compa

## Стратегия и применение

**Порядок операций:**
1. Парсим `salary_range` (строки вида `"50000-80000"`) в `salary_min` / `salary_max` — ДО любых дропов, иначе при >70% NaN колонка умерла бы без парсинга.
2. Считаем доли NaN и дропаем колонки с >70% пропусков (включая `salary_range` после того, как из неё извлечены числа — дубль не нужен).
3. Числовые: clip по перцентилям [1, 99] → impute median. 0 в `salary_*` не заполняем нулём, медиана корректна.
4. `CAT_LOW` (<20): impute mode → one-hot (`pd.get_dummies`, dummy_na=False).
5. `CAT_HIGH` / `CAT_MID` (>20): frequency encoding = count / total (NaN считается отдельной категорией 'NaN' при подсчёте, но сначала impute mode).
6. Текстовые — оставляем как есть; NaN заменяем на пустую строку, чтобы DS не падал на `.str` операциях.
7. Target (`fraudulent`) — не трогаем.
8. Сохраняем в `cleaned.csv`.

In [ ]:
import os

df = DF.copy()

actions = []  # column, strategy, reason

# ---- 1. Парсинг salary_range до дропа ----
if 'salary_range' in df.columns:
    def _parse_sal(x):
        if pd.isna(x):
            return (np.nan, np.nan)
        s = str(x).strip()
        parts = s.split('-')
        if len(parts) != 2:
            return (np.nan, np.nan)
        try:
            a = float(parts[0]); b = float(parts[1])
            return (a, b)
        except Exception:
            return (np.nan, np.nan)
    parsed = df['salary_range'].apply(_parse_sal)
    df['salary_min'] = parsed.apply(lambda t: t[0])
    df['salary_max'] = parsed.apply(lambda t: t[1])
    actions.append(('salary_range', 'parse → salary_min/salary_max, then drop original',
                    'сырая строка "min-max", извлекаем числа ДО drop по NaN'))
    df = df.drop(columns=['salary_range'])

# Пересчитываем группы после парсинга
TEXT_COLS = [c for c in ['title', 'company_profile', 'description', 'requirements', 'benefits']
             if c in df.columns]
NUM_COLS = [c for c in df.columns
            if c != TARGET and c not in TEXT_COLS
            and pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = [c for c in df.columns
            if c != TARGET and c not in TEXT_COLS and c not in NUM_COLS
            and (pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c]))]

# ---- 2. Drop колонок с >70% NaN ----
nan_ratio = df.isna().mean()
to_drop = [c for c in df.columns if c != TARGET and nan_ratio[c] > 0.70]
for c in to_drop:
    actions.append((c, 'drop_column', f'NaN ratio {nan_ratio[c]:.2%} > 70%'))
df = df.drop(columns=to_drop)

NUM_COLS  = [c for c in NUM_COLS  if c in df.columns]
CAT_COLS  = [c for c in CAT_COLS  if c in df.columns]
TEXT_COLS = [c for c in TEXT_COLS if c in df.columns]

# ---- 3. Числовые: clip [1,99] + median impute ----
for c in NUM_COLS:
    if c == TARGET:
        continue
    lo = df[c].quantile(0.01)
    hi = df[c].quantile(0.99)
    if pd.notna(lo) and pd.notna(hi) and lo != hi:
        df[c] = df[c].clip(lower=lo, upper=hi)
        actions.append((c, f'clip to [p01={lo:.2f}, p99={hi:.2f}]', 'контроль выбросов'))
    med = df[c].median()
    n_nan = df[c].isna().sum()
    if n_nan > 0:
        df[c] = df[c].fillna(med)
        actions.append((c, f'impute median={med}', f'распределение скошено, было {n_nan} NaN'))

# ---- 4/5. Категориальные ----
CAT_LOW  = [c for c in CAT_COLS if df[c].nunique(dropna=True) < 20]
CAT_HIGH_MID = [c for c in CAT_COLS if c not in CAT_LOW]

# mode-impute всем категориальным
for c in CAT_COLS:
    n_nan = df[c].isna().sum()
    if n_nan > 0:
        mode_val = df[c].mode(dropna=True)
        fill_val = mode_val.iloc[0] if len(mode_val) else 'unknown'
        df[c] = df[c].fillna(fill_val)
        actions.append((c, f'impute mode="{fill_val}"', f'категориальная, было {n_nan} NaN'))

# one-hot для CAT_LOW
if CAT_LOW:
    before_cols = df.shape[1]
    df = pd.get_dummies(df, columns=CAT_LOW, prefix=CAT_LOW, dummy_na=False)
    for c in CAT_LOW:
        actions.append((c, 'one-hot encoding', f'<20 уникальных значений'))
    print(f"one-hot: +{df.shape[1]-before_cols} новых колонок для {CAT_LOW}")

# frequency encoding для остальных
total = len(df)
for c in CAT_HIGH_MID:
    nunq = df[c].nunique(dropna=True)
    freq = df[c].value_counts(dropna=False) / total
    df[c] = df[c].map(freq).astype(float)
    actions.append((c, 'frequency encoding (count/total)',
                    f'{nunq} уникальных — one-hot взорвал бы размерность'))

# ---- 6. Текстовые: NaN → "" ----
for c in TEXT_COLS:
    n_nan = df[c].isna().sum()
    if n_nan > 0:
        df[c] = df[c].fillna('')
        actions.append((c, 'fill NaN with empty string; text as_is',
                        f'длинный текст, {n_nan} NaN; фичеризацию оставляем DS'))
    else:
        actions.append((c, 'as_is', 'длинный текст, фичеризацию оставляем DS'))

# ---- 7. Target не трогаем (проверка) ----
assert TARGET in df.columns, 'target дропнут — ошибка!'
actions.append((TARGET, 'keep as_is', 'target, никаких трансформаций'))

# ---- 8. Save ----
out_path = "/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df.to_csv(out_path, index=False)

print("cleaned shape:", df.shape)
print("total NaN in cleaned:", df.isna().sum().sum())
print("target preserved:", TARGET in df.columns)
print("saved to:", out_path)

ACTIONS_DF = pd.DataFrame(actions, columns=['column', 'strategy', 'reason'])

one-hot: +22 новых колонок для ['employment_type', 'required_experience', 'required_education']
cleaned shape: (17880, 39)
total NaN in cleaned: 0
target preserved: True
saved to: /Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv


## Применённые действия

Таблица column / strategy / reason ниже (печатается из `ACTIONS_DF`).

In [ ]:
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)
print(ACTIONS_DF.to_string(index=False))

             column                                          strategy                                                 reason
       salary_range parse → salary_min/salary_max, then drop original сырая строка "min-max", извлекаем числа ДО drop по NaN
         salary_min                                       drop_column                                 NaN ratio 84.11% > 70%
         salary_max                                       drop_column                                 NaN ratio 84.11% > 70%
             job_id                clip to [p01=179.79, p99=17701.21]                                      контроль выбросов
      telecommuting                      clip to [p01=0.00, p99=1.00]                                      контроль выбросов
   has_company_logo                      clip to [p01=0.00, p99=1.00]                                      контроль выбросов
      has_questions                      clip to [p01=0.00, p99=1.00]                                      контроль выбросов
